# Event Labeling

This notebook converts residual threshold crossings into supervised learning labels. The target is whether a residual reverts before a stop-loss barrier inside a fixed holding period.

## Research setup

The label must match the trading question. The model will later estimate the probability that a candidate mean-reversion trade succeeds before the stop-loss, conditional on information available at entry time. The label is forward-looking, so it belongs only in the target column, not in the feature set.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.labeling import (
    LabelingConfig,
    generate_event_labels,
    label_distribution,
    summarize_event_labels,
    success_rate_by_z_bucket,
)
from src.database import (
    connect_database,
    initialize_database,
    store_candidate_events,
    store_event_labels,
    store_event_label_summary,
)

ROOT = Path('..')
DATA = ROOT / 'data' / 'processed'
FIGURES = ROOT / 'figures'
SQL_SCHEMA = ROOT / 'sql' / 'schema.sql'
DATABASE = ROOT / 'data' / 'database' / 'triangular_stat_arb.sqlite'
DATA.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## Residual source

The preferred input is a dynamic residual table produced by the rolling, ridge, or Kalman hedge-ratio methods. If a residual z-score column already exists, the labeler uses it. Otherwise, the z-score is estimated using prior observations only.

In [ ]:
def load_residual_source() -> pd.DataFrame:
    candidates = [
        DATA / 'kalman_residual_table.csv',
        DATA / 'dynamic_residual_table.csv',
        DATA / 'residual_table.csv',
    ]
    for path in candidates:
        if path.exists():
            frame = pd.read_csv(path, parse_dates=['date'])
            if 'residual' in frame.columns:
                return frame
    return pd.read_csv(DATA / 'candidate_events_synthetic_residuals.csv', parse_dates=['date'])


residuals = load_residual_source()
residuals.head()

## Labeling parameters

The entry threshold defines candidate events. The exit threshold defines residual reversion. The stop-loss threshold defines residual widening. The maximum holding period prevents indefinite labels.

In [ ]:
config = LabelingConfig(
    entry_z=2.0,
    exit_z=0.5,
    stop_loss_z=3.0,
    max_holding_period=15,
    z_window=60,
    min_periods=20,
)

outputs = generate_event_labels(residuals, config=config)
scored_residuals = outputs['scored_residuals']
candidate_events = outputs['candidate_events']
event_labels = outputs['event_labels']

candidate_events.head(), event_labels.head()

## Summary tables

The success rate should be interpreted as a label diagnostic. It is not a strategy performance estimate because it excludes costs, position sizing, capacity, and portfolio interaction.

In [ ]:
triplet_summary = summarize_event_labels(event_labels)
z_bucket_summary = success_rate_by_z_bucket(event_labels, buckets=[2.0, 2.5, 3.0, 3.5, 4.0, 10.0])

candidate_events.to_csv(DATA / 'candidate_events_table.csv', index=False)
event_labels.to_csv(DATA / 'event_labels_table.csv', index=False)
triplet_summary.to_csv(DATA / 'event_success_rate_by_triplet.csv', index=False)
z_bucket_summary.to_csv(DATA / 'event_success_rate_by_z_bucket.csv', index=False)

triplet_summary

In [ ]:
z_bucket_summary

## Label distribution

Class imbalance matters because a model can appear accurate by mostly predicting the majority class. Downstream evaluation should use precision, recall, calibration, and walk-forward performance rather than raw accuracy alone.

In [ ]:
distribution = label_distribution(event_labels)
fig, ax = plt.subplots(figsize=(8, 5))
if distribution.empty:
    ax.text(0.5, 0.5, 'No candidate events', ha='center', va='center')
    ax.set_axis_off()
else:
    ax.bar(distribution['outcome'], distribution['n_events'])
    ax.set_title('Event Label Distribution')
    ax.set_xlabel('Outcome')
    ax.set_ylabel('Number of events')
fig.tight_layout()
fig.savefig(FIGURES / 'event_label_distribution.png', dpi=160)

## SQL storage

The event tables preserve the candidate entry conditions and the forward-looking outcome separately. This separation makes it easier to audit feature leakage in the machine learning stages downstream.

In [ ]:
initialize_database(DATABASE, SQL_SCHEMA)
with connect_database(DATABASE) as conn:
    store_candidate_events(conn, candidate_events, if_exists='replace')
    store_event_labels(conn, event_labels, if_exists='replace')
    store_event_label_summary(conn, triplet_summary, if_exists='replace')